# ts_aos_analysis

# AOS  DM-46763:  WET-007 Compare CWFS approaches with OR4 ComCam data 

For https://rubinobs.atlassian.net/browse/SITCOM-1149

Last verified to run 2024/10/24

Versions:

* lsst_distrib w_2024_37 (ext, cvmfs)

* ts_wep v11.5.2

Use AOS OR4 LsstComCamSim  data. It is in `embargo_or4` repository. The planner for observations was https://github.com/jmeyers314/aos_block_planner/blob/main/OR4.ipynb  

In this notebook we run ISR, donut detection and cutouts, and TIE / Danish Zernike retrieval. The main objective is to verify whether TIE / Danish provide similar fit results.


## Imports

In [ ]:
from lsst.daf import butler as dafButler
import matplotlib.pyplot as plt 
from astropy.visualization import ZScaleInterval
import numpy as np
from astropy.table import Table
import itertools
from bokeh.palettes import viridis
from bokeh.plotting import figure, show
from bokeh.plotting import output_notebook

## Inspect the data

First, find out the range of data for the first OR4 night:

In [ ]:
butler = dafButler.Butler('/repo/embargo')

In [ ]:
butler = dafButler.Butler('/repo/embargo')
refs = list(butler.registry.queryDatasets('raw',  collections=['LSSTComCam/raw/all'],
                              where="instrument='LSSTComCam' and day_obs=20241024 and exposure.observation_type='cwfs'"
                             ).expanded())

In [ ]:
for ref in refs[:100]:
    print(ref.dataId.exposure.observation_type,  ref.dataId.detector.id, ref.dataId.exposure.observation_reason,  ref.dataId.exposure.seq_num,  ref.dataId.exposure.science_program)

In [ ]:
refs[0].dataId.exposure.observation_type = 'cwfs

In [ ]:
butler = dafButler.Butler('embargo_or4')
len(list(butler.registry.queryDatasets('raw', collections=['LSSTComCamSim/defaults'],
                                        where= "instrument='LSSTComCamSim' and day_obs=20240625").expanded()))

Check how in this particular simulation the `observation_type` connects to `seq_num`: 

In [ ]:
dataRefs = butler.registry.queryDatasets('raw', collections=['LSSTComCamSim/defaults'],
                                         where= "day_obs=20240625 and instrument='LSSTComCamSim'").expanded()

for ref in list(dataRefs)[58:80]:
    print(ref.dataId.exposure.id, ref.dataId.exposure.observation_type, ref.dataId.exposure.seq_num,
         ref.dataId.exposure.observation_reason, ref.dataId.exposure.science_program)
    

Each intra/extra pair (`cwfs`) is followed by an in-focus exposure (`acq`). The first in each sequence is intra-focal, extra-focal, then in-focus.  All Zernikes are calculated based such intra/extra pair, and stored under the extra-focal seqNum (eg. 62). Thus showing Zk fit for seqNum 62, which used exposures with  seqNum 61 and 62.

How many datasets have `observation_type` of `cwfs`:

In [ ]:
len(list(butler.registry.queryDatasets('raw', collections=['LSSTComCamSim/defaults'],
         where= "day_obs=20240625 and exposure.observation_type='cwfs' and instrument='LSSTComCamSim'").expanded()))


Run these through ISR pipeline:

    cd /sdf/group/rubin/shared/scichris/DM-46763_WET-007

    
    allocateNodes.py -v -n 10 -c 64 -m 60:00:00 -q milano -g 1800 s3df --account rubin:developers
    

    bps submit site_bps.yaml \
    -b embargo_or4 \
    -i LSSTComCamSim/defaults \
    -o u/scichris/or4_night1_isr \
    -p lsstComCamSimPipelineISR.yaml \
    -d "day_obs=20240625 and exposure.observation_type='cwfs' and instrument='LSSTComCamSim'"



Check the ISR results:

In [ ]:
butler = dafButler.Butler('embargo_or4')
dataRefs = butler.registry.queryDatasets('postISRCCD', collections=['u/scichris/or4_night1_isr']).expanded()
refs = []
for ref in dataRefs: 
    refs.append(ref)
print(len(refs))

In [ ]:
dataRefs = butler.registry.queryDatasets('postISRCCD', collections=['u/scichris/or4_night1_isr'],
                                        where="instrument='LSSTComCamSim'  and exposure in (7024062500061)").expanded()

In [ ]:
len(list(dataRefs))

In [ ]:
list(dataRefs)

That shows that indeed all the `raw` datasets now are avaiable as `postISRCCD`. Show R22 postISRCCDs:

In [ ]:
exps = [butler.get('postISRCCD', dataId=ref.dataId, collections=['u/scichris/or4_night1_isr']) for ref in list(dataRefs)]


In [ ]:

zscale = ZScaleInterval()

fig,axs = plt.subplots(3,3,figsize=(10,10))
ax = np.ravel(axs)
for i in range(len(exps)):
    exp=exps[i]
    d = exp.image.array
    vmin,vmax = zscale.get_limits(d)
    ax[i].imshow(d, vmin=vmin, vmax=vmax, origin='lower')
    #ax.set_title(f'{refs[0].dataId.exposure.id}, det {detId} ')
    #ax.set_xlabel('x [px]')
    #ax.set_ylabel('y [px]')
    ax[i].set_xticks([])
    ax[i].set_yticks([])
fig.subplots_adjust(hspace=0.01, wspace=0.02)

Run donut detection and cutouts with `bps` 

    bps submit site_bps.yaml \
    -b embargo_or4 \
    -i u/scichris/or4_night1_isr \
    -o u/scichris/or4_night1_direct_stamps_WEP_12-2 \
    -p lsstComCamSimPipelineDirectCutoutOnly.yaml \
    -d " exposure.observation_type='cwfs' and instrument='LSSTComCamSim'"
    

Check that the donut cutouts make sense: 

In [ ]:
butler = dafButler.Butler('embargo_or4')
donutStampsIntra = butler.get('donutStampsIntra', dataId=list(dataRefs)[3].dataId,
                              collections=['u/scichris/or4_night1_direct_stamps_WEP_12-2'])

In [ ]:
import numpy as np 
fig,axs = plt.subplots(6,6)

ax = np.ravel(axs)

for i  in range(len(ax)):
    stamp = donutStampsIntra[i]
    ax[i].imshow(stamp.stamp_im.image.array, origin='lower')
if len(donutStampsIntra)<len(ax):
    for i in range(len(donutStampsIntra), len(ax)):
        ax[i].axis('off')
        

We used `donutSelector.useCustomMagLimit: True` to provide as many donuts as possible.

## Run the pipeline from donut detection to Zernikes

The pipeline yaml contains:


    # Here we specify the corresponding instrument for the data we
    # will be using.
    instrument: lsst.obs.lsst.LsstComCamSim
    
    # Then we can specify each task in our pipeline by a name
    # and then specify the class name corresponding to that task
    tasks:
      generateDonutDirectDetectTask:
        class: lsst.ts.wep.task.generateDonutDirectDetectTask.GenerateDonutDirectDetectTask
        config:
          donutSelector.useCustomMagLimit: True
      cutOutDonutsScienceSensorGroupTask:
        class: lsst.ts.wep.task.cutOutDonutsScienceSensorTask.CutOutDonutsScienceSensorTask
        config:
          donutStampSize: 160
          initialCutoutPadding: 40
          python: |
            from lsst.ts.wep.task.pairTask import GroupPairer
            config.pairer.retarget(GroupPairer)
      calcZernikesTask:
        class: lsst.ts.wep.task.calcZernikesTask.CalcZernikesTask
        config:
          python: |
            from lsst.ts.wep.task import  EstimateZernikesDanishTask
            config.estimateZernikes.retarget(EstimateZernikesDanishTask)
      aggregateZernikeTablesTask: lsst.donut.viz.AggregateZernikeTablesTask
      aggregateDonutTablesTask: lsst.donut.viz.AggregateDonutTablesTask
      aggregateAOSVisitTableTask: lsst.donut.viz.AggregateAOSVisitTableTask
      plotAOSTask: lsst.donut.viz.PlotAOSTask
      aggregateDonutStampsTask: lsst.donut.viz.AggregateDonutStampsTask
      plotDonutTask: lsst.donut.viz.PlotDonutTask
            
(`config.estimateZernikes.retarget(EstimateZernikesDanishTask)`  for the Danish version )

    allocateNodes.py -v -n 10 -c 64 -m 60:00:00 -q roma -g 1800 s3df --account rubin:developers
    
    bps submit site_bps.yaml \
    -b embargo_or4 \
    -i u/scichris/or4_night1_isr  \
    -o u/scichris/or4_night1_direct_danish_WEP_12-2_test \
    -p lsstComCamSimPipelineDirectToZkGroupingDanish.yaml \
    -d " exposure.observation_type='cwfs' and instrument='LSSTComCamSim'"

     
    bps submit site_bps.yaml \
    -b embargo_or4 \
    -i u/scichris/or4_night1_direct_stamps_WEP_12-2 \
    -o u/scichris/or4_night1_direct_tie_WEP_12-2_test \
    -p lsstComCamSimPipelineDirectToZkGrouping.yaml  \
    -d "exposure.observation_type='cwfs' and instrument='LSSTComCamSim'"
    

## Compare Danish and TIE fit results:

We load the results from butler and ensure that there is the same number of  Zernike estimates  from Danish as TIE. The data product that is equivalent to `zernikeEstimateAvg` is `aggregateZernikesAvg`:

In [ ]:
butler = dafButler.Butler('embargo_or4')

In [ ]:
butler.registry.queryDatasetTypes('*donut*')

In [ ]:
method='tie'
collection = f'u/scichris/or4_night1_direct_{method}_WEP_12-2_test'
dataRefs = butler.registry.queryDatasets('donutTable',collections=[collection]).expanded()


In [ ]:
list(dataRefs)

In [ ]:
butler = dafButler.Butler('embargo_or4')
dataRefsMethod={}
for method in ['tie', 'danish']:
    collection = f'u/scichris/or4_night1_direct_{method}_WEP_12-2_test'
    dataRefs = butler.registry.queryDatasets('aggregateZernikesAvg',collections=[collection]).expanded()
    dataRefsMethod[method]=dataRefs
assert len(list(dataRefsMethod['tie'])) == len(list(dataRefsMethod['danish'])) 


Load all results to a dictionary for quicker plotting:

In [ ]:
# prepare  the dictionary
results = {}
for method in ['danish','tie']:
    results[method] = {}
    for visit in np.unique(visits):
        results[method][visit] = {}

for ref in dataRefs:
    visit = ref.dataId.visit.id
    for method in results.keys():
        collection =  f'u/scichris/or4_night1_direct_{method}_WEP_12-2_test'
        results[method][visit] =    butler.get('aggregateZernikesAvg', dataId=ref.dataId, collections=[collection])
                                   

Plot the two methods for all detectors from a single visit: 

In [ ]:
fig,axs = plt.subplots(3,3,figsize=(10,8))
ax = np.ravel(axs)
for method in results.keys():   
    aggregate = results[method][visit]
    i=0
    for row in aggregate.iterrows():
        ax[i].plot(np.arange(4,29),row[0], ls='-.', label=method)
        ax[i].set_title(f'detector {row[1]}')
        ax[i].set_xticks(range(4,29,4))
        i += 1 
ax[2].legend(bbox_to_anchor=[1.0,0.5])
fig.subplots_adjust(hspace=0.4, wspace=0.2)
fig.suptitle(f'OR4 night1, visit{visit}')
fig.text(0.5,0.05, 'Zk mode')
fig.text(0.05,0.5, 'Zk value [microns]', rotation='vertical')

Show whether there is much difference between TIE and Danish for all defocal pairs (all `cwfs` visits):

In [ ]:
fig,axs = plt.subplots(3,3,figsize=(10,8))
ax = np.ravel(axs)
visits = results['tie'].keys()
for visit in visits:
    aggregateTie = results['tie'][visit]
    aggregateDanish = results['danish'][visit]

    for i in range(len(ax)):
        tieFit = aggregateTie['zk_CCS'][i]
        danishFit = aggregateDanish['zk_CCS'][i]
        diff = danishFit-tieFit   
        ax[i].plot(np.arange(4,29), diff, ls='-.', alpha=0.5, label=visit)
        ax[i].set_xticks(range(4,29,4))
ax[2].legend(bbox_to_anchor=[1.0,0.5])
fig.subplots_adjust(hspace=0.4, wspace=0.2)
fig.suptitle(f'OR4 night1, cwfs visits, range {min(visits)}:{max(visits)}')
fig.text(0.5,0.05, 'Zk mode')
fig.text(0.05,0.5, f'$\Delta$ zk fit (Danish-TIE) [microns]', rotation='vertical')

If we plot that as absolute difference, we'll notice that most of the exposures have agreement within 0.1 microns:

In [ ]:
aggregateTie

In [ ]:
fig,axs = plt.subplots(3,3,figsize=(10,8))
ax = np.ravel(axs)
visits = results['tie'].keys()
for visit in visits:
    aggregateTie = results['tie'][visit]
    aggregateDanish = results['danish'][visit]

    for i in range(len(ax)):
        tieFit = aggregateTie['zk_CCS'][i]
        danishFit = aggregateDanish['zk_CCS'][i]
        diff = danishFit-tieFit   
        ax[i].plot(np.arange(4,29), abs(diff), ls='-.', alpha=0.5, label=visit)
        ax[i].set_xticks(range(4,29,4))
ax[2].legend(bbox_to_anchor=[1.0,0.5])
fig.subplots_adjust(hspace=0.4, wspace=0.2)
fig.suptitle(f'OR4 night1, cwfs visits, range {min(visits)}:{max(visits)}')
fig.text(0.5,0.05, 'Zk mode')
fig.text(0.05,0.5, f'|$\Delta$ zk fit| (Danish-TIE) [microns]', rotation='vertical')



We can plot the differences for a single detector interactively so that it's easier to focus on an individual visit. Here we color-coded by the value of the RMS difference between TIE and Danish fit result:

In [ ]:
output_notebook()
visits = list(results['danish'].keys())
palette = viridis(len(visits))
# plot a single detector 
i=0
ax = figure(width=800, height=500,title=f'OR4 night1, cwfs visits, range {min(visits)}:{max(visits)}, TIE vs Danish',
                    x_axis_label=r'$$ $$Zk  mode',
                    y_axis_label=r'|$$\Delta$$| zk fit (Danish-TIE) [microns]',)

# calculate rmss to sort 
rmss=[]
diffs = []
for visit in visits :
    aggregateTie = results['tie'][visit]
    aggregateDanish = results['danish'][visit]
    tieFit = aggregateTie['zk_CCS'][i] 
    danishFit = aggregateDanish['zk_CCS'][i]
    diff = danishFit-tieFit
    diffs.append(abs(diff))
    rmss.append(np.sqrt(np.mean(np.square(diff))))
    
args = np.argsort(rmss)
                
# first plot fit results 
for i in range(len(visits)):
    ax.line(np.arange(4,29),diffs[i], line_width=3,
            color=palette[args[i]],
            legend_label=str(visits[i])
           )
    
ax.legend.location = "top_right"
ax.legend.click_policy="mute"
ax.legend.ncols=2
show(ax)

This allows us to interactively see which visits are most disparate between the two methods (in this case, `seqnum` 110).